# Week 06 — ML Model Integration and Inference
## LeafGuard AI: Understanding and Serving the Plant Disease Model

This notebook explains how the ML model works and how to integrate it into the FastAPI backend.

**Learning outcomes:**
- Understand the model-as-a-function concept (input contract → output contract)
- Implement image preprocessing for CNN models
- Load a TensorFlow/Keras model and run inference
- Interpret prediction output arrays
- Understand transfer learning with MobileNetV2
- Convert a Keras model to TFLite format

---

## Cell 1: Install ML Dependencies


In [ ]:
# Install required packages
# !pip install tensorflow Pillow numpy matplotlib

import numpy as np
import matplotlib
import PIL

try:
    import tensorflow as tf
    print(f'TensorFlow version: {tf.__version__}')
    print(f'Keras version: {tf.keras.__version__}')
    gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
    print(f'GPU available: {gpu_available}')
except ImportError:
    print('TensorFlow not installed.')
    print('Install with: pip install tensorflow')
    print('For this notebook, we will demonstrate with numpy arrays instead.')

print(f'NumPy version: {np.__version__}')
print(f'Pillow version: {PIL.__version__}')
print('Ready!')

## Cell 2: The Model as a Function

A trained CNN model is essentially a mathematical function:
```
f(image_tensor) → probability_array
```

You don't need to understand how the function was learned (gradient descent, backpropagation).
You need to understand the **input/output contract**.

In [ ]:
import numpy as np

# Simulate what a CNN model does
# Input: (1, 224, 224, 3) float tensor normalised to [0,1]
# Output: (1, 38) softmax probabilities summing to 1.0

def simulate_model_predict(input_tensor):
    """Simulate a trained plant disease CNN model output."""
    # Real model uses learned weights to compute class scores
    # Here we simulate with random logits + softmax for demonstration
    num_classes = 38
    np.random.seed(42)  # Fixed seed for reproducibility
    raw_logits = np.random.randn(num_classes)
    # Softmax: convert logits to probabilities summing to 1.0
    exp_logits = np.exp(raw_logits - np.max(raw_logits))  # Numerically stable
    probabilities = exp_logits / exp_logits.sum()
    return probabilities.reshape(1, -1)

# Simulate input image tensor
input_tensor = np.random.rand(1, 224, 224, 3).astype(np.float32)

print('INPUT CONTRACT:')
print(f'  Shape: {input_tensor.shape}  [batch, height, width, channels]')
print(f'  Dtype: {input_tensor.dtype}')
print(f'  Range: [{input_tensor.min():.4f}, {input_tensor.max():.4f}]')
print(f'  Meaning: batch=1 image, 224×224 pixels, 3 colour channels (RGB)')

# Run prediction
output = simulate_model_predict(input_tensor)

print()
print('OUTPUT CONTRACT:')
print(f'  Shape: {output.shape}  [batch, num_classes]')
print(f'  Dtype: {output.dtype}')
print(f'  Values: probabilities for each of 38 disease classes')
print(f'  Sum: {output[0].sum():.6f} (should be exactly 1.0 — softmax property)')
print(f'  Max confidence: {output[0].max():.4f} at index {output[0].argmax()}')
print()
print('Top 5 predictions (random model — ignore actual disease names):')
top5_idx = np.argsort(output[0])[::-1][:5]
for i, idx in enumerate(top5_idx, 1):
    print(f'  #{i}: Class {idx:2d} → {output[0][idx]*100:.2f}%')

## Cell 3: PlantVillage Class Names

The model predicts one of 38 classes from the PlantVillage dataset.

In [ ]:
import numpy as np

# All 38 PlantVillage class names (used in labels.txt)
CLASS_NAMES = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew',
    'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch',
    'Strawberry___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

print(f'Total classes: {len(CLASS_NAMES)}')
print()

# Analyse class distribution
plants = {}
for name in CLASS_NAMES:
    plant = name.split('___')[0]
    plants[plant] = plants.get(plant, 0) + 1

print('Disease classes per plant type:')
for plant, count in sorted(plants.items(), key=lambda x: -x[1]):
    bar = '█' * count
    healthy_count = sum(1 for n in CLASS_NAMES if plant in n and 'healthy' in n)
    disease_count = count - healthy_count
    print(f'  {plant:<35} {bar} ({disease_count} disease + {healthy_count} healthy)')

print()
healthy_classes = [n for n in CLASS_NAMES if 'healthy' in n]
disease_classes = [n for n in CLASS_NAMES if 'healthy' not in n]
print(f'Healthy classes: {len(healthy_classes)}')
print(f'Disease classes: {len(disease_classes)}')

# Check if predicted class is healthy
def is_healthy_prediction(class_name):
    return 'healthy' in class_name.lower()

print()
print('Is Tomato___Early_blight healthy?', is_healthy_prediction('Tomato___Early_blight'))
print('Is Tomato___healthy healthy?', is_healthy_prediction('Tomato___healthy'))

## Cell 4: Image Preprocessing Pipeline

The most critical step — if preprocessing doesn't match training, accuracy collapses.

In [ ]:
from PIL import Image
import numpy as np
import io

def preprocess_image_for_cnn(image_bytes: bytes, target_size=(224, 224)) -> np.ndarray:
    """
    Complete preprocessing pipeline for plant disease CNN.

    Input:  Raw image bytes (JPEG/PNG from HTTP upload or file)
    Output: (1, 224, 224, 3) float32 tensor, normalised to [0, 1]
    """
    # Step 1: Decode bytes to PIL Image
    image = Image.open(io.BytesIO(image_bytes))

    # Step 2: Ensure RGB (handles RGBA, grayscale, palette-based)
    image = image.convert('RGB')

    # Step 3: Resize to model input size
    # LANCZOS = high-quality downsampling filter
    image = image.resize(target_size, Image.LANCZOS)

    # Step 4: Convert to NumPy float32 array
    img_array = np.array(image, dtype=np.float32)
    # Shape is now (224, 224, 3)

    # Step 5: Normalise pixel values from [0, 255] to [0.0, 1.0]
    img_array = img_array / 255.0

    # Step 6: Add batch dimension — model expects (batch, H, W, C)
    img_array = np.expand_dims(img_array, axis=0)
    # Shape is now (1, 224, 224, 3)

    return img_array

# Create a synthetic leaf image for testing
def create_synthetic_leaf(width=512, height=400, with_disease=True):
    """Create a synthetic leaf image for testing preprocessing."""
    img = Image.new('RGB', (width, height), color=(34, 139, 34))  # Green leaf
    pixels = np.array(img, dtype=np.uint8)

    # Add leaf veins
    for i in range(height):
        # Central vein
        pixels[i, width//2-2:width//2+2, :] = [0, 100, 0]
    for j in range(width):
        # Horizontal veins
        if j % 40 == 0:
            pixels[height//2-1:height//2+1, j:j+width//2, :] = [0, 100, 0]

    if with_disease:
        # Add brown spots (simulate Early Blight)
        import random
        random.seed(42)
        for _ in range(15):
            cx = random.randint(50, width-50)
            cy = random.randint(50, height-50)
            r = random.randint(10, 30)
            for y in range(max(0, cy-r), min(height, cy+r)):
                for x in range(max(0, cx-r), min(width, cx+r)):
                    dist = ((x-cx)**2 + (y-cy)**2)**0.5
                    if dist < r:
                        # Concentric ring effect (brown → yellow → brown)
                        intensity = dist / r
                        if intensity < 0.5:
                            pixels[y, x, :] = [80, 50, 20]   # Dark brown centre
                        else:
                            pixels[y, x, :] = [180, 170, 30]  # Yellow halo

    return Image.fromarray(pixels)

# Test the preprocessing pipeline
leaf_image = create_synthetic_leaf(with_disease=True)

# Convert to bytes (simulating an HTTP upload)
buffer = io.BytesIO()
leaf_image.save(buffer, format='JPEG', quality=95)
image_bytes = buffer.getvalue()

# Run preprocessing
processed = preprocess_image_for_cnn(image_bytes)

print('=== PREPROCESSING PIPELINE ===')
print(f'Input:  {len(image_bytes):,} bytes (JPEG)')
print(f'Original size: {leaf_image.width}×{leaf_image.height} pixels')
print(f'Output shape: {processed.shape}')
print(f'Output dtype: {processed.dtype}')
print(f'Pixel range: [{processed.min():.4f}, {processed.max():.4f}]')
print(f'Mean pixel: {processed.mean():.4f}')
print()
print('Channel statistics (R, G, B):')
for i, channel in enumerate(['Red', 'Green', 'Blue']):
    ch = processed[0, :, :, i]
    print(f'  {channel}: mean={ch.mean():.4f}, std={ch.std():.4f}')

## Cell 5: Transfer Learning with MobileNetV2

MobileNetV2 was pre-trained on ImageNet (1.2M images, 1000 classes).
We repurpose it for 38 plant disease classes.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras import layers, Model

    NUM_CLASSES = 38

    # Step 1: Load MobileNetV2 pre-trained on ImageNet, without the classification head
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,      # Remove ImageNet 1000-class softmax layer
        weights='imagenet'      # Use ImageNet pre-trained weights
    )

    print('Base model summary:')
    print(f'  Total layers: {len(base_model.layers)}')
    print(f'  Trainable params: {base_model.count_params():,}')
    print(f'  Input shape: {base_model.input_shape}')
    print(f'  Output shape: {base_model.output_shape}')

    # Step 2: Freeze the pre-trained weights
    base_model.trainable = False

    # Step 3: Build custom classification head
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)   # Reduce (7,7,1280) → (1280,)
    x = layers.Dropout(0.2)(x)               # Prevent overfitting
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs, outputs)

    print()
    print('Full model (base + classification head):')
    print(f'  Total layers: {len(model.layers)}')
    print(f'  Total params: {model.count_params():,}')
    trainable = sum(layer.count_params() for layer in model.layers if layer.trainable)
    print(f'  Trainable params: {trainable:,}  (only the new head)')
    print(f'  Non-trainable: {model.count_params() - trainable:,}  (frozen MobileNetV2)')

    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print()
    print('Model compiled and ready for training!')
    print('Run: model.fit(train_dataset, epochs=10, validation_data=val_dataset)')

except ImportError:
    print('TensorFlow not installed — showing conceptual diagram instead')
    print()
    diagram = '''
    MobileNetV2 Architecture for LeafGuard AI:

    Input (224×224×3 normalised image)
    ↓
    MobileNetV2 stem (conv 3×3, stride 2) → 112×112×32
    ↓
    17× Inverted residual blocks (depthwise + pointwise conv) → 7×7×1280
    ↓
    [FROZEN — pre-trained on ImageNet, NOT retrained]
    ↓
    GlobalAveragePooling2D → 1280
    ↓
    Dropout(0.2)
    ↓
    Dense(38, softmax) → 38 probabilities  [TRAINABLE — our custom head]
    ↓
    Output: [0.01, 0.02, ..., 0.91, ..., 0.01]  (one per disease class)
    '''
    print(diagram)

## Cell 6: Decoding Model Output

The model outputs raw probabilities. This cell shows how to decode them into meaningful predictions.

In [ ]:
import numpy as np

CLASS_NAMES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy'
]

def format_class_name(raw_name: str) -> str:
    """Convert PlantVillage class name to human-readable format."""
    parts = raw_name.split('___')
    plant = parts[0].replace('_', ' ').replace('(', '').replace(')', '').strip()
    disease = parts[1].replace('_', ' ').strip() if len(parts) > 1 else 'Unknown'
    if disease.lower() == 'healthy':
        return f'{plant} (Healthy)'
    return f'{plant}: {disease}'

def decode_prediction(probabilities: np.ndarray, class_names: list, top_k: int = 5) -> dict:
    """Decode model output array into a structured prediction result."""
    # Remove batch dimension if present
    if probabilities.ndim == 2:
        probabilities = probabilities[0]

    # Top predicted class
    top_idx = int(np.argmax(probabilities))
    top_confidence = float(probabilities[top_idx])
    top_class = class_names[top_idx]

    # Top K predictions
    top_k_indices = np.argsort(probabilities)[::-1][:top_k]
    top_k_results = [
        {
            'rank': i + 1,
            'class': class_names[idx],
            'display_name': format_class_name(class_names[idx]),
            'confidence': float(probabilities[idx]),
            'confidence_pct': f'{probabilities[idx]*100:.1f}%'
        }
        for i, idx in enumerate(top_k_indices)
    ]

    return {
        'top_prediction': {
            'class': top_class,
            'display_name': format_class_name(top_class),
            'confidence': top_confidence,
            'is_healthy': 'healthy' in top_class.lower(),
            'is_uncertain': top_confidence < 0.5
        },
        'top_5': top_k_results
    }

# Simulate a model output for a tomato with early blight (class index 29)
np.random.seed(7)
raw_probs = np.random.dirichlet(np.ones(38) * 0.1)  # Mostly noise
# Boost the Early Blight class to simulate a confident prediction
raw_probs[29] = 0.91  # Tomato___Early_blight
raw_probs /= raw_probs.sum()  # Renormalise

result = decode_prediction(raw_probs, CLASS_NAMES)

print('=== MODEL OUTPUT DECODING ===')
print()
print('Top prediction:')
for k, v in result['top_prediction'].items():
    print(f'  {k}: {v}')

print()
print('Top 5 predictions:')
for pred in result['top_5']:
    bar = '█' * int(pred['confidence'] * 40)
    print(f"  #{pred['rank']}: {pred['display_name']:<45} {bar} {pred['confidence_pct']}")

print()
if result['top_prediction']['is_uncertain']:
    print('⚠️  Low confidence — model is uncertain. Suggest retaking the photo.')
elif result['top_prediction']['is_healthy']:
    print('✅ Plant appears healthy!')
else:
    print(f"⚠️  Disease detected: {result['top_prediction']['display_name']}")

## Cell 7: TFLite Conversion

To run inference on Android without internet, we convert the Keras model to TFLite format.

In [ ]:
try:
    import tensorflow as tf
    import tempfile
    import os

    print('=== TFLITE CONVERSION PROCESS ===')
    print()

    # Demonstrate conversion (using a tiny model for speed)
    # In practice, use the full plant disease model
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(5, activation='softmax')
    ])

    # ---- OPTION 1: Default conversion (FP32 weights) ----
    converter = tf.lite.TFLiteConverter.from_keras_model(demo_model)
    tflite_fp32 = converter.convert()
    print(f'Option 1 — FP32 (no quantization):')
    print(f'  Size: {len(tflite_fp32):,} bytes ({len(tflite_fp32)/1024:.1f} KB)')

    # ---- OPTION 2: Dynamic range quantization (INT8 weights) ----
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant = converter.convert()
    print(f'Option 2 — INT8 quantization:')
    print(f'  Size: {len(tflite_quant):,} bytes ({len(tflite_quant)/1024:.1f} KB)')
    reduction = (1 - len(tflite_quant)/len(tflite_fp32)) * 100
    print(f'  Size reduction: {reduction:.1f}%')

    print()
    print('For the real plant disease model (~20 MB Keras):')
    print('  FP32 TFLite:  ~20 MB')
    print('  INT8 TFLite:  ~5 MB (75% reduction)')
    print()

    # ---- RUN TFLite INFERENCE ----
    interpreter = tf.lite.Interpreter(model_content=tflite_fp32)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print('TFLite Interpreter Tensor Details:')
    print(f'  Input tensor:  name={input_details[0]["name"]}, shape={input_details[0]["shape"]}, dtype={input_details[0]["dtype"].__name__}')
    print(f'  Output tensor: name={output_details[0]["name"]}, shape={output_details[0]["shape"]}, dtype={output_details[0]["dtype"].__name__}')

    # Run inference
    test_input = np.random.rand(1, 10).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    print()
    print('Inference result:')
    print(f'  Output shape: {output.shape}')
    print(f'  Probabilities: {output[0]}')
    print(f'  Sum: {output[0].sum():.6f}')

except ImportError:
    print('TensorFlow not installed.')
    print()
    print('TFLite conversion commands (run in terminal with TF installed):')
    print('''
import tensorflow as tf

# Load your trained model
model = tf.keras.models.load_model("plant_disease_model.h5")

# Convert to TFLite with INT8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save to file
with open("model/plant_disease.tflite", "wb") as f:
    f.write(tflite_model)
    
print(f"TFLite model size: {len(tflite_model)/1024/1024:.1f} MB")
    ''')

## Cell 8: model_loader.py Walkthrough


In [ ]:
import os

# Read the actual model_loader.py from the repository
model_loader_path = os.path.join('..', '..', 'backend-api', 'model_loader.py')

try:
    with open(model_loader_path, 'r') as f:
        content = f.read()
    print(content)
except FileNotFoundError:
    print('model_loader.py not found at relative path.')
    print('Run this notebook from: notebooks/week-06/')
    print()
    print('The complete inference pipeline is in backend-api/model_loader.py')

## Cell 9: Checkpoint Exercises

1. ☐ Create a synthetic test image and run it through `preprocess_image_for_cnn()`
2. ☐ Modify `decode_prediction()` to also return the plant type separately from the disease name
3. ☐ Add a confidence threshold parameter — return "uncertain" if confidence < threshold
4. ☐ Run the decode function on an "all-equal" probability array — what happens?
5. ☐ (If TF installed) Load MobileNetV2 and count how many layers are frozen vs trainable
6. ☐ (If TF installed) Convert the demo model to TFLite and run inference

**Bonus:** Add a function that groups the top-5 predictions by plant type and shows confidence per plant.

## Summary

| Concept | Key point |
|---------|----------|
| Model contract | Input: (1, 224, 224, 3) float32 [0,1]. Output: (1, 38) softmax |
| Preprocessing | Resize → RGB → normalise → add batch dim |
| Transfer learning | MobileNetV2 base (frozen) + custom Dense head (trainable) |
| Decoding output | argmax for top class, interpret healthy vs disease |
| TFLite | Quantised mobile format, ~75% smaller, same accuracy |

---
*See also: `roadmap/week-06-cloud-ml-model/learning-notes.md`*  
*And: `model/model-acquisition-guide.md` for training your own model*